# Trabalho Prático de Mineração de Dados — Fase 2
## Utilização de LLMs e Análise Comparativa com Diretrizes de IA Responsável

**Disciplina:** Mineração de Dados  
**Grupo:**  João Marcos Oliveira Neves (2024008644), Júlia Borba Fonseca de Souza (2023117580), Laura Martins Froede (2023087877), Matheus Soares dos Santos de Freitas (2024080043)  
**Dataset:** Credit Card Dataset for Clustering
**Tarefa de mineração:** Agrupamento
**Data da entrega:** 24/05/2026

---

## Objetivo deste notebook

Este notebook documenta a Fase 2 do Trabalho Prático. O objetivo é usar uma ou mais LLMs como apoio à construção da solução de mineração de dados e comparar duas trilhas de interação:

- **Trilha A — baseline:** interação com a LLM sem explicitar diretrizes de IA responsável;
- **Trilha B — guiada:** interação com a LLM explicitando as diretrizes de IA responsável escolhidas pelo grupo.

O foco não é apenas mostrar prompts e respostas. O grupo deve analisar criticamente se a explicitação das diretrizes levou a diferenças concretas na solução proposta, por exemplo em:

- escolha de algoritmos;
- tratamento dos dados;
- seleção de atributos;
- definição de parâmetros;
- métricas de avaliação;
- interpretabilidade;
- privacidade;
- análise de viés;
- custo computacional;
- qualidade da documentação.

> **Importante:** este notebook é um modelo de estrutura. Ele contém exemplos, placeholders e código genérico. O grupo deve substituir os campos indicados por informações reais do seu dataset, das interações com LLMs e dos experimentos efetivamente realizados.


# Como usar este notebook

Este notebook foi organizado seguindo a estrutura esperada para a Fase 2:

1. **Business Understanding**
2. **Data Understanding & Data Preparation**
3. **Modeling**
4. **Evaluation**
5. **Checklist final**

Ao longo do notebook, os trechos marcados entre colchetes, como `[INSERIR LINK DA CONVERSA]`, devem ser preenchidos pelo grupo.

## Diferença entre exemplo e entrega real

- Células de **exemplo** mostram como organizar a entrega.
- Células com **placeholders** indicam pontos que devem ser preenchidos.
- Células de **código** são genéricas e podem precisar de adaptação ao dataset.
- Nenhum resultado experimental deve ser inventado.
- Toda conclusão empírica deve estar apoiada em uma tabela, gráfico ou execução de código.


# 0. Controle da entrega e rastreabilidade

Esta seção registra informações mínimas para tornar o trabalho rastreável e auditável.

| Item | Valor |
|---|---|
| Nome do dataset | Credit Card Dataset for Clustering |
| Link público do dataset | https://www.kaggle.com/datasets/arjunbhasin2013/ccdata |
| Número de linhas | 8950 |
| Número de colunas | 18 |
| Tarefa de mineração | Agrupamento |
| LLM usada na Trilha A | Gemini Pro |
| LLM usada na Trilha B | Gemini Pro |
| Link da conversa — Trilha A | [INSERIR LINK DA CONVERSA] |
| Link da conversa — Trilha B | [INSERIR LINK DA CONVERSA] |

## Diretrizes de IA responsável escolhidas

| Diretriz | Justificativa da escolha | Como será explicitada na Trilha B |
|---|---|---|
| Explicabilidade | Algoritmos de clustering dividem o espaço multidimensional de forma puramente matemática. Sem uma tradução semântica, os perfis gerados tornam-se caixas-pretas estatísticas inutilizáveis para equipes de risco e de crédito humanas. | A instrução de modelagem exigirá: "Atue como um Especialista em Ciências Comportamentais Aplicadas a Finanças. Dado o vetor de centroides de cada cluster gerado, traduza as médias numéricas em personas financeiras detalhadas (ex: ’O Consumidor Consciente Parcelador’ ou ’O Usuário Dependente de Saques Emergenciais’), justificando o perfil com base nas variáveis estruturais." |
| Justiça e Viés | Agrupamentos baseados em saldos devedores e capacidade de pagamento mínimo podem criar loops de feedback discriminatórios. Se um cluster rotular sistematicamente indivíduos de menor poder aquisitivo em grupos de "altíssimo risco", o algoritmo pode justificar a redução automática de seus limites, agravando sua vulnerabilidade econômica. | A LLM será guiada pela seguinte restrição: "Atue como um ombudsman de IA Ética. Analise a distribuição de limites e saldos nos clusters gerados. Avalie se o modelo está penalizando desproporcionalmente grupos com menor capacidade de pagamento com rótulos de risco excessivos e sugira formas de balancear a interpretação do cluster para evitar a exclusão financeira injusta." |

> **Propósito pedagógico:** esta seção evita que a análise fique solta ou apenas narrativa. O avaliador precisa conseguir identificar o que foi perguntado à LLM, quando, em qual modelo e com qual objetivo.


In [1]:
# Configurações iniciais do notebook
# Esta célula centraliza imports e parâmetros gerais.
# O grupo deve adaptar os caminhos, nomes de colunas e tipo de tarefa.

import os
import time
import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Caminho para o dataset.
# Exemplos:
# DATA_PATH = "dados/dataset.csv"
# DATA_PATH = "/content/dataset.csv"
DATA_PATH = "https://raw.githubusercontent.com/Doctor-Math/TPs-Mineracao-de-Dados/VariableMath/data/data_tp2/CC%20GENERAL.xls"

# Tipo de tarefa do TP.
# Opções sugeridas:
# - "padroes_frequentes"
# - "agrupamento"
# - "classificacao"
TASK_TYPE = "Agrupamento"

# Coluna-alvo, se aplicável.
# Para padrões frequentes ou agrupamento, normalmente pode ser None.
TARGET_COLUMN = None  # Exemplo: "classe"

# Coluna ou conjunto de colunas usadas para gerar transações, se o TP for de padrões frequentes.
TRANSACTION_COLUMN = "[INSERIR_COLUNA_DE_TRANSACOES]"

print("Configuração carregada.")
print(f"Tarefa definida: {TASK_TYPE}")


Configuração carregada.
Tarefa definida: Agrupamento


# 1. Business Understanding

Nesta seção, o grupo deve explicar o problema, o contexto dos dados, a relevância da tarefa e a hipótese experimental sobre o uso das diretrizes de IA responsável.

A seção deve responder:

1. Qual problema será investigado?
2. Por que o dataset é relevante?
3. Qual tarefa de mineração de dados será executada?
4. Qual é o valor esperado da análise?
5. Quais diretrizes de IA responsável foram escolhidas?
6. Por que essas diretrizes são pertinentes ao problema?
7. O que se espera que mude quando essas diretrizes são explicitadas à LLM?


## 1.1 Descrição do problema de negócio

- **Contexto:** Instituições financeiras gerenciam milhões de titulares de cartões de crédito ativos, gerando um volume massivo de dados transacionais diários. Em vez de analisar os clientes apenas por métricas isoladas (como o score de crédito tradicional), o banco precisa entender a dinâmica comportamental multidimensional de seus usuários, como a relação entre gastos à vista, parcelamentos, saques emergenciais e o ritmo de pagamento das faturas ao longo do tempo.
- **Problema:** A análise agregada ou baseada em regras estáticas falha em capturar os perfis sutis de consumo e de saúde financeira dos clientes. Sem uma segmentação inteligente e não supervisionada, a instituição financeira não consegue diferenciar o cliente que usa o limite alto com responsabilidade daquele que está operando no limite devido ao superendividamento, o que gera ineficiência na oferta de produtos e aumento no risco de inadimplência (default).
- **Possíveis interessados:**
  - Equipe de Gestão de Risco de Crédito: Interessada em antecipar perfis de risco de endividamento antes que ocorra a inadimplência.
  - Time de Marketing e Produtos Financeiros: Interessado em criar campanhas personalizadas e ajustar benefícios do cartão (milhas, seguros, cashback) para os perfis certos.
  - Analistas de Experiência do Cliente (CX): Focados em identificar clientes insatisfeitos ou subutilizados para ações de engajamento.
- **Decisões apoiadas pela análise:**
  - Realocação automatizada e preventiva de limites de crédito (aumento para perfis saudáveis e redução ou congelamento para perfis de risco).
  - Personalização de políticas de taxas de juros para parcelamentos baseada no histórico comportamental do cluster.
  - Intervenções proativas de saúde financeira (alertas e ofertas de refinanciamento amigável para clusters em vias de superendividamento).
- **Limitações iniciais conhecidas:** O dataset compreende um histórico restrito de apenas 6 meses de atividade, o que impede a análise de sazonalidades anuais longas (como compras de fim de ano). Além disso, a base é estritamente comportamental e anonimizada, não contendo variáveis demográficas essenciais (como idade, profissão ou renda declarada), o que exige que a renda e o contexto do cliente sejam inferidos indiretamente a partir do saldo e do volume de gastos (proxy features).

> Este texto deve ser escrito pelo grupo. Não copie a descrição do dataset sem contextualizar o problema de mineração de dados.


## 1.2 Objetivo do dataset, origem e características gerais

### Objetivo do dataset

O propósito principal deste conjunto de dados é permitir a segmentação de clientes bancários por meio de seus históricos de consumo e pagamento. Ele foi estruturado especificamente para tarefas de aprendizado não supervisionado (clustering), servindo para agrupar usuários com perfis transacionais semelhantes e identificar anomalias ou comportamentos de risco no uso do cartão de crédito.

### Origem dos dados

Os dados são reais e provêm de uma instituição financeira operante no mercado global, tendo sido anonimizados e disponibilizados para fins acadêmicos e de desenvolvimento de modelos. A base é amplamente conhecida e hospedada na plataforma Kaggle.

Link público: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

### Características do dataset

| Coluna | Tipo esperado | Descrição | Relevância para a tarefa |
|---|---|---|---|
| CUST_ID | object (string) | Identificador único do titular do cartão (mascarado). | Nula. Será removida ou indexada, pois não possui valor preditivo ou comportamental. |
| BALANCE | float32 | Saldo devedor/pendente na conta do cliente para compras. | Altíssima. Indica o montante que o cliente carrega como dívida ativa de um mês para o outro. |
| BALANCE_FREQUENCY | float32 | Frequência com que o saldo é atualizado (relação entre 0 e 1). | Alta. Mede a constância do uso do cartão. Próximo a 1 significa uso diário/contínuo. |
| PURCHASES | float32 | Valor total de compras realizadas nos últimos 6 meses. | Altíssima. Principal indicador do volume de consumo do cliente. |
| ONEOFF_PURCHASES | float32 | Valor máximo de transações feitas em uma única compra (à vista). | Média. Ajuda a diferenciar compradores de itens de alto valor daqueles de gastos fracionados. |
| INSTALLMENTS_PURCHASES | float32 | Valor total de compras realizadas de forma parcelada. | Altíssima. Essencial para testar a Hipótese 1 (separação entre parceladores e compradores à vista). |
| CASH_ADVANCE | float32 | Valor total de saques em dinheiro utilizando o limite do cartão. | Altíssima. Principal indicador de comportamento de risco e busca por liquidez emergencial (Hipótese 2). |
| PURCHASES_FREQUENCY | float32 | Frequência com que as compras são feitas (relação entre 0 e 1). | Alta. Diferencia o comprador assíduo do usuário esporádico. |
| ONEOFF_PURCHASES_FREQUENCY | float32 | Frequência de compras à vista (relação entre 0 e 1). | Média. Mede a adesão ao comportamento de pagamento imediato. |
| PURCHASES_INSTALLMENTS_FREQ | float32 | Frequência de compras parceladas (relação entre 0 e 1). | Média. Mede a dependência do parcelamento no dia a dia. |
| CASH_ADVANCE_FREQUENCY | float32 | Frequência com que são realizados saques em dinheiro. | Alta. Avalia se o comportamento de risco por saques é crônico ou um evento isolado. |
| CASH_ADVANCE_TRX | int32 | Número de transações de saque em dinheiro realizadas. | Média. Quantifica a recorrência de saques de forma discreta. |
| PURCHASES_TRX | int32 | Número de transações de compra realizadas. | Alta. Permite entender o tíquete médio das compras quando cruzada com PURCHASES. |
| CREDIT_LIMIT | float32 | Limite de crédito total concedido ao cliente. | Altíssima. Crucial para avaliar a capacidade financeira e testar a Hipótese 3 (uso do teto disponível). |
| PAYMENTS | float32 | Valor total de pagamentos efetuados pelo cliente nos 6 meses. | Altíssima. Mede a capacidade do cliente de amortizar sua dívida real com o banco. |
| MINIMUM_PAYMENTS | float32 | Valor mínimo de pagamento exigido pela fatura. | Altíssima. Possui dados faltantes (missing values) e indica a propensão ao endividamento rotativo. |
| PRC_FULL_PAYMENT | float32 | Percentual da fatura pago integralmente pelo cliente. | Alta. Identifica os clientes financeiramente saudáveis (que quitam 100% da fatura constantemente). |
| TENURE | int32 | Tempo de relacionamento do cliente com o cartão (em meses). | Baixa. Controla o tempo de observação do comportamento (geralmente fixado em 6 meses nesta base). |


### Relação com o problema de negócio

Este conjunto de dados é perfeito para o problema de negócio proposto porque cobre simultaneamente as três dimensões críticas do comportamento em cartões de crédito: Consumo (PURCHASES), Endividamento (BALANCE, CASH_ADVANCE) e Amortização (PAYMENTS, MINIMUM_PAYMENTS).

Ao contrário de uma base sintética (que geraria padrões perfeitos e artificiais), este dataset possui as imperfeições de um cenário bancário real: distribuições severamente assimétricas (poucos clientes gastam milhões, enquanto a maioria gasta pouco), forte correlação linear entre variáveis (como frequências de compra e volumes gastos) e dados ausentes em contas recém-criadas ou inativas. Isso exigirá o uso rigoroso de técnicas de preparação de dados (como normalização por escala e engenharia de atributos) e permitirá avaliar de forma justa o impacto e a eficiência dos algoritmos K-Means e DBSCAN no Google Colab.

## 1.3 Diretrizes selecionadas e hipótese experimental

### Diretrizes selecionadas

| Diretriz | Por que é pertinente ao problema? | Possível efeito esperado na solução da LLM |
|---|---|---|
| Explicabilidade | Algoritmos de agrupamento (clustering) operam dividindo o espaço matemático multidimensional de forma abstrata. Sem uma tradução semântica dos centroides e das distâncias, as "personas financeiras" geradas tornam-se caixas-pretas estatísticas inutilizáveis por equipes de risco e analistas de negócio humanos. | Espera-se que a LLM guie o grupo na interpretação qualitativa dos centroides, transformando médias puras de colunas como CASH_ADVANCE e INSTALLMENTS_PURCHASES em descrições claras de comportamento (personas), sugerindo também métodos visuais (como mapas de calor e gráficos de coordenadas paralelas) para justificar os grupos gerados. |
| Justiça e Mitigação de Viés | Segmentações baseadas em histórico de adimplência, saldos devedores e pagamento mínimo podem criar loops de discriminação indireta (proxy discrimination). Se o modelo isolar e rotular de forma puramente punitiva grupos de baixa renda sob a alcunha de "altíssimo risco", pode induzir o banco a decisões automatizadas de redução severa de limites, agravando a vulnerabilidade econômica desses clientes. | Espera-se que a LLM alerte sobre o perigo de rotulagem estigmatizante sobre variáveis de capacidade de pagamento, propondo uma avaliação de impacto de equidade e sugerindo restrições éticas na interpretação dos clusters para evitar decisões de exclusão financeira sistemática. |

### Hipótese experimental

Esperamos que, ao explicitar as diretrizes de Explicabilidade e Justiça e Mitigação de Viés, a LLM proponha na trilha guiada uma estratégia de modelagem com maior interpretabilidade analítica (focada na construção e validação de personas financeiras transparentes) e maior cuidado com impactos regulatórios e discriminação indireta, em comparação com a solução baseline (que tende a sugerir apenas métricas e algoritmos matemáticos puros, sem contextualização ética ou de negócios).

### Evidências que serão analisadas

Para avaliar a hipótese, compararemos as trilhas quanto a:
- **algoritmos sugeridos**: se a trilha guiada sugere algoritmos que facilitem a interpretação de centroides (como K-Means) ou detecção de anomalias com justificativa ética (como DBSCAN);
- **etapas de preparação dos dados**: se há inclusão de técnicas para monitorar ou mitigar disparidades de escala que possam enviesar o agrupamento contra perfis específicos;
- **parâmetros recomendados**: como a LLM orienta a escolha do número de clusters ($k$) visando o equilíbrio entre a qualidade estatística e a explicabilidade prática para o negócio;
- **métricas propostas**: se além das métricas internas de agrupamento (Silhueta, Davies-Bouldin), são propostas métricas de validação de viés ou de interpretabilidade das personas;
- **justificativas técnicas**: a profundidade dos argumentos usados para validar a escolha de uma partição espacial específica;riscos apontados: se a LLM mapeia os perigos éticos de loops de feedback discriminatórios causados pela segmentação automatizada;
- **limitações reconhecidas**: o reconhecimento de que perfis demográficos e socioeconômicos ocultos podem estar sofrendo vieses indiretos no modelo;
- **aderência às diretrizes escolhidas**: o grau em que as respostas deixam de ser meros relatórios de código e passam a atuar como um guia de IA Responsável.


## 1.4 Interações com LLM — Business Understanding

### Trilha A — baseline

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]  
**Subtarefa:** obter apoio para formular o Business Understanding.

#### Prompt principal — baseline

```text
Estou fazendo um trabalho de Mineração de Dados com o dataset CC GENERAL (Credit Card Dataset para Clustering).
A tarefa principal é aplicar técnicas de agrupamento (Clustering), especificamente extração de padrões frequentes e engenharia de atributos, para segmentar clientes de cartão de crédito e definir estratégias de marketing.

Ajude-me a formular o Business Understanding (Entendimento do Negócio), incluindo o objetivo do uso desse dataset, a origem dos dados, suas características gerais e a relação direta com o problema de negócio de uso de crédito.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE DA RESPOSTA. NÃO COPIAR RESPOSTA LONGA INTEGRALMENTE SE NÃO FOR NECESSÁRIO.]

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]  
**Subtarefa:** obter apoio para formular o Business Understanding considerando as diretrizes escolhidas.

#### Prompt principal — guiado

```text
Estou fazendo um trabalho de Mineração de Dados com o dataset CC GENERAL.
A tarefa principal é aplicar técnicas de agrupamento (Clustering) e mineração de padrões frequentes para segmentação de comportamento de clientes de cartões de crédito para fins de marketing.
Ajude-me a formular o Business Understanding.

Considere explicitamente as seguintes diretrizes de IA responsável:
1. Privacidade e Proteção de Dados: Como a análise de saldos (BALANCE), compras (PURCHASES) e saques em dinheiro (CASH_ADVANCE) reflete hábitos financeiros sensíveis dos indivíduos, discuta limites para que a engenharia de atributos não gere perfis excessivamente invasivos ou exponha vulnerabilidades econômicas.
2. Equidade e Prevenção de Discriminação Excludente: Garanta que os clusters gerados para campanhas de marketing ou concessão de benefícios não atuem como um viés punitivo contra perfis de menor renda ou com baixos pagamentos mínimos (MINIMUM_PAYMENTS), perpetuando barreiras econômicas de forma injusta.

Além da descrição do problema, indique riscos éticos, limitações analíticas das variáveis financeiras fornecidas e critérios quantitativos/qualitativos de sucesso que deveriam orientar a solução de mineração de dados.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE DA RESPOSTA]


## 1.5 Análise crítica das trilhas — Business Understanding

| Critério | Trilha A — baseline | Trilha B — guiada | Diferença observada |
|---|---|---|---|
| Clareza do problema | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Relação com o dataset | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Consideração das diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Identificação de riscos | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da justificativa | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Erros ou omissões | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Síntese crítica

[INSERIR ANÁLISE COMPARATIVA]

A resposta guiada deve ser considerada melhor apenas se houver evidência concreta de melhoria. Exemplos de evidência:

- mencionou atributos sensíveis que a baseline ignorou;
- propôs critérios de avaliação mais adequados;
- apontou limitações metodológicas relevantes;
- sugeriu documentação mais rastreável;
- evitou recomendações inadequadas feitas na baseline.

Caso não haja diferença relevante, isso também deve ser relatado.


# 2. Data Understanding & Data Preparation

Nesta seção, o grupo deve explorar o dataset, identificar problemas de qualidade dos dados e preparar os dados para a etapa de modelagem.

Além disso, deve registrar como a LLM foi usada para apoiar:

- exploração inicial;
- identificação de problemas;
- tratamento de valores ausentes;
- transformação de atributos;
- seleção de variáveis;
- preparação específica para a tarefa de mineração;
- identificação de aspectos relacionados às diretrizes de IA responsável.


In [6]:
import pandas as pd

import urllib.request
from pathlib import Path

def carregar_dataset(caminho):
    # 1. Verifica se é um link da internet
    if str(caminho).startswith(("http://", "https://")):
        # Se for link, o pandas consegue ler direto sem precisar do pathlib
        if str(caminho).lower().endswith(".csv"):
            return pd.read_csv(caminho)
        elif str(caminho).lower().endswith((".xls", ".xlsx")):
            return pd.read_excel(caminho)
        else:
            raise ValueError("Formato de URL não tratado.")
            
    # 2. Se não for link, trata como arquivo local (seu código original)
    caminho_path = Path(caminho)
    if not caminho_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
        
    if caminho_path.suffix.lower() == ".csv":
        return pd.read_csv(caminho_path)
    elif caminho_path.suffix.lower() in [".xls", ".xlsx"]:
        return pd.read_excel(caminho_path)
    else:
        raise ValueError("Formato não tratado neste modelo. Adapte a função carregar_dataset.")

# Truque: Vamos salvar localmente forçando a extensão .csv, porque o arquivo REALMENTE é um CSV
NOME_ARQUIVO_LOCAL = "CC_GENERAL.csv" 

print("Baixando o arquivo...")
urllib.request.urlretrieve(DATA_PATH, NOME_ARQUIVO_LOCAL)
print("Download concluído!")

# Agora sua função vai entrar no 'if' do CSV e ler perfeitamente!
df = carregar_dataset(NOME_ARQUIVO_LOCAL)

print("\nDataset carregado com sucesso!")
print(f"Número de linhas: {df.shape[0]}")
print(f"Número de colunas: {df.shape[1]}")
display(df.head())

Baixando o arquivo...
Download concluído!

Dataset carregado com sucesso!
Número de linhas: 8950
Número de colunas: 18


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [7]:
# Verificação mínima das restrições do dataset
# A especificação exige dataset público, com pelo menos 1000 linhas e 4 colunas.
# O link público deve ser documentado em Markdown; aqui validamos apenas dimensões.

def verificar_restricoes_dataset(df: pd.DataFrame) -> None:
    n_linhas, n_colunas = df.shape

    print("Verificação das dimensões do dataset")
    print(f"- Linhas: {n_linhas}")
    print(f"- Colunas: {n_colunas}")

    if n_linhas < 1000:
        print("ATENÇÃO: o dataset possui menos de 1000 linhas.")
    else:
        print("OK: o dataset possui pelo menos 1000 linhas.")

    if n_colunas < 4:
        print("ATENÇÃO: o dataset possui menos de 4 colunas.")
    else:
        print("OK: o dataset possui pelo menos 4 colunas.")

# Executar após carregar o dataset:
verificar_restricoes_dataset(df)


Verificação das dimensões do dataset
- Linhas: 8950
- Colunas: 18
OK: o dataset possui pelo menos 1000 linhas.
OK: o dataset possui pelo menos 4 colunas.


In [8]:
# Exploração inicial
# Esta célula gera uma visão geral do dataset.
# O grupo deve interpretar os resultados em uma célula Markdown logo abaixo.

def resumo_inicial(df: pd.DataFrame) -> pd.DataFrame:
    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "perc_nulos": (df.isna().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True)
    })

    return resumo.sort_values(by="perc_nulos", ascending=False)

# Exemplo de uso:
resumo = resumo_inicial(df)
display(resumo)


,tipo,n_nulos,perc_nulos,n_unicos
MINIMUM_PAYMENTS,float64,313,3.50,8636
CREDIT_LIMIT,float64,1,0.01,205
BALANCE,float64,0,0.00,8871
CUST_ID,object,0,0.00,8950
BALANCE_FREQUENCY,float64,0,0.00,43
PURCHASES,float64,0,0.00,6203
CASH_ADVANCE,float64,0,0.00,4323
PURCHASES_FREQUENCY,float64,0,0.00,47
ONEOFF_PURCHASES,float64,0,0.00,4014
INSTALLMENTS_PURCHASES,float64,0,0.00,4452


In [9]:
# Estatísticas descritivas
# O objetivo é separar análise de variáveis numéricas e categóricas.

def estatisticas_descritivas(df: pd.DataFrame):
    numericas = df.select_dtypes(include=np.number)
    categoricas = df.select_dtypes(exclude=np.number)

    print("Colunas numéricas:", list(numericas.columns))
    print("Colunas não numéricas:", list(categoricas.columns))

    if len(numericas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis numéricas")
        display(numericas.describe().T)

    if len(categoricas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis categóricas/textuais")
        display(categoricas.describe().T)

# Exemplo de uso:
estatisticas_descritivas(df)


Colunas numéricas: ['BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']
Colunas não numéricas: ['CUST_ID']

Estatísticas descritivas — variáveis numéricas


,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.474828,2081.531879,0.000000,128.281915,873.385231,2054.140036,19043.13856
BALANCE_FREQUENCY,8950.0,0.877271,0.236904,0.000000,0.888889,1.000000,1.000000,1.00000
PURCHASES,8950.0,1003.204834,2136.634782,0.000000,39.635000,361.280000,1110.130000,49039.57000
ONEOFF_PURCHASES,8950.0,592.437371,1659.887917,0.000000,0.000000,38.000000,577.405000,40761.25000
INSTALLMENTS_PURCHASES,8950.0,411.067645,904.338115,0.000000,0.000000,89.000000,468.637500,22500.00000
CASH_ADVANCE,8950.0,978.871112,2097.163877,0.000000,0.000000,0.000000,1113.821139,47137.21176
PURCHASES_FREQUENCY,8950.0,0.490351,0.401371,0.000000,0.083333,0.500000,0.916667,1.00000
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.202458,0.298336,0.000000,0.000000,0.083333,0.300000,1.00000
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.364437,0.397448,0.000000,0.000000,0.166667,0.750000,1.00000
CASH_ADVANCE_FREQUENCY,8950.0,0.135144,0.200121,0.000000,0.000000,0.000000,0.222222,1.50000



Estatísticas descritivas — variáveis categóricas/textuais


,count,unique,top,freq
CUST_ID,8950,8950,C10001,1


## 2.1 Exploração inicial — interpretação

[INSERIR ANÁLISE DA EXPLORAÇÃO INICIAL]

A análise deve comentar, no mínimo:

- quantidade de linhas e colunas;
- tipos de atributos;
- valores ausentes;
- variáveis com muitos valores distintos;
- possíveis identificadores;
- possíveis atributos sensíveis;
- variáveis relevantes para a tarefa de mineração;
- limitações iniciais percebidas.

### Relação com as diretrizes escolhidas

[EXPLICAR COMO A EXPLORAÇÃO INICIAL SE RELACIONA ÀS DIRETRIZES]

Exemplos:

- Se a diretriz for **Privacidade e Segurança**, verificar atributos identificadores ou sensíveis.
- Se a diretriz for **Justiça e Viés**, verificar grupos sub-representados ou classes desbalanceadas.
- Se a diretriz for **Eficiência e Escalabilidade**, verificar dimensionalidade, cardinalidade e volume dos dados.
- Se a diretriz for **Explicabilidade**, verificar se os atributos são interpretáveis.


In [ ]:
# Visualizações exploratórias básicas
# O grupo deve adaptar as colunas às características do dataset.

def plot_distribuicao_numerica(df: pd.DataFrame, coluna: str) -> None:
    plt.figure(figsize=(8, 4))
    df[coluna].dropna().hist(bins=30)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

def plot_contagem_categorica(df: pd.DataFrame, coluna: str, top_n: int = 20) -> None:
    plt.figure(figsize=(10, 4))
    df[coluna].value_counts(dropna=False).head(top_n).plot(kind="bar")
    plt.title(f"Contagem de valores — {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

def plot_matriz_correlacao(df: pd.DataFrame) -> None:
    numericas = df.select_dtypes(include=np.number)

    if numericas.shape[1] < 2:
        print("Não há pelo menos duas variáveis numéricas para calcular correlação.")
        return

    corr = numericas.corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Matriz de correlação — variáveis numéricas")
    plt.tight_layout()
    plt.show()

# Exemplos de uso:
# plot_distribuicao_numerica(df, "[COLUNA_NUMERICA]")
# plot_contagem_categorica(df, "[COLUNA_CATEGORICA]")
# plot_matriz_correlacao(df)


## 2.2 Análise visual — interpretação

[INSERIR INTERPRETAÇÃO DOS GRÁFICOS]

A interpretação deve evitar apenas descrever o óbvio. Procure responder:

- Há concentração de valores?
- Há valores extremos?
- Há categorias muito raras?
- Há desbalanceamento?
- Os padrões observados afetam a tarefa de mineração?
- Alguma visualização sugere risco associado às diretrizes escolhidas?

> Não invente conclusões. Toda afirmação empírica deve estar apoiada em uma tabela, gráfico ou cálculo executado no notebook.


In [ ]:
# Detecção simples de outliers em variáveis numéricas usando IQR
# Esta função é um exemplo. O grupo deve avaliar se o critério faz sentido para cada atributo.

def detectar_outliers_iqr(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    serie = df[coluna].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    print(f"Coluna: {coluna}")
    print(f"Q1: {q1}")
    print(f"Q3: {q3}")
    print(f"IQR: {iqr}")
    print(f"Limite inferior: {limite_inferior}")
    print(f"Limite superior: {limite_superior}")
    print(f"Número de outliers: {len(outliers)}")

    return outliers

# Exemplo de uso:
# outliers = detectar_outliers_iqr(df, "[COLUNA_NUMERICA]")
# display(outliers.head())


## 2.3 Problemas de qualidade dos dados

| Problema identificado | Evidência | Decisão tomada | Justificativa |
|---|---|---|---|
| Valores ausentes em [COLUNA] | [TABELA/GRÁFICO/CÁLCULO] | [REMOVER/IMPUTAR/MANTER] | [JUSTIFICAR] |
| Duplicatas | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Outliers em [COLUNA] | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Categorias raras | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Atributos sensíveis ou identificadores | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |

### Observação sobre responsabilidade

[EXPLICAR SE ALGUMA DECISÃO DE PREPARAÇÃO TEM IMPLICAÇÕES PARA AS DIRETRIZES ESCOLHIDAS]

Exemplo:

> A remoção de um atributo sensível pode reduzir risco de exposição indevida, mas também pode dificultar a análise de disparidades entre grupos. Essa decisão deve ser justificada com cuidado.


In [ ]:
# Exemplo de preparação dos dados
# Esta célula é propositalmente genérica.
# O grupo deve adaptar de acordo com o dataset e a tarefa.

def preparar_dados_basico(
    df: pd.DataFrame,
    colunas_remover=None,
    imputar_numericas=True,
    imputar_categoricas=True
) -> pd.DataFrame:
    df_prep = df.copy()

    # 1. Remover duplicatas exatas
    df_prep = df_prep.drop_duplicates()

    # 2. Remover colunas explicitamente marcadas como irrelevantes ou identificadoras
    if colunas_remover is None:
        colunas_remover = [
            # "[INSERIR_COLUNA_IDENTIFICADORA]",
            # "[INSERIR_COLUNA_IRRELEVANTE]",
        ]

    colunas_existentes = [col for col in colunas_remover if col in df_prep.columns]
    df_prep = df_prep.drop(columns=colunas_existentes)

    # 3. Exemplo de tratamento simples de nulos
    # Atenção: imputação deve ser justificada no texto.
    if imputar_numericas:
        for col in df_prep.select_dtypes(include=np.number).columns:
            df_prep[col] = df_prep[col].fillna(df_prep[col].median())

    if imputar_categoricas:
        for col in df_prep.select_dtypes(include=["object", "category"]).columns:
            df_prep[col] = df_prep[col].fillna("desconhecido")

    return df_prep

# Exemplo de uso:
# df_prep = preparar_dados_basico(df, colunas_remover=["id"])
# display(df_prep.head())


## 2.4 Justificativa da preparação dos dados

[INSERIR JUSTIFICATIVA DAS TRANSFORMAÇÕES REALIZADAS]

Explique:

- quais colunas foram removidas e por quê;
- como valores ausentes foram tratados;
- como variáveis categóricas foram tratadas;
- se houve normalização, discretização ou codificação;
- se houve remoção de outliers;
- quais riscos essas decisões introduzem;
- como essas decisões se relacionam às diretrizes escolhidas.

> A preparação dos dados não deve ser apenas uma sequência de comandos. Toda decisão relevante precisa de justificativa técnica.


## 2.5 Interações com LLM — Data Understanding & Data Preparation

### Subtarefa comparável

Nesta etapa, as duas trilhas devem receber a mesma subtarefa geral.

**Subtarefa escolhida:** [EXEMPLO: propor estratégias de limpeza, transformação e preparação dos dados para a tarefa de mineração]

---

### Trilha A — baseline

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — baseline

```text
Tenho um dataset de cartões de crédito (CC GENERAL) com as seguintes colunas: CUST_ID, BALANCE, BALANCE_FREQUENCY, PURCHASES, ONEOFF_PURCHASES, INSTALLMENTS_PURCHASES, CASH_ADVANCE, PURCHASES_FREQUENCY, ONEOFF_PURCHASES_FREQUENCY, PURCHASES_INSTALLMENTS_FREQUENCY, CASH_ADVANCE_FREQUENCY, CASH_ADVANCE_TRX, PURCHASES_TRX, CREDIT_LIMIT, PAYMENTS, MINIMUM_PAYMENTS, PR_FULL_PAYMENT, TENURE.

A tarefa de mineração é aplicar o algoritmo K-Means para realizar o agrupamento (Clustering) e segmentação dos perfis de comportamento financeiro dos clientes.

Sugira uma estratégia de Data Understanding e Data Preparation para preparar os dados para essa tarefa.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — guiado

```text
Tenho um dataset de cartões de crédito (CC GENERAL) com as seguintes colunas: CUST_ID, BALANCE, BALANCE_FREQUENCY, PURCHASES, ONEOFF_PURCHASES, INSTALLMENTS_PURCHASES, CASH_ADVANCE, PURCHASES_FREQUENCY, ONEOFF_PURCHASES_FREQUENCY, PURCHASES_INSTALLMENTS_FREQUENCY, CASH_ADVANCE_FREQUENCY, CASH_ADVANCE_TRX, PURCHASES_TRX, CREDIT_LIMIT, PAYMENTS, MINIMUM_PAYMENTS, PR_FULL_PAYMENT, TENURE.

A tarefa de mineração é aplicar o algoritmo K-Means para agrupar e segmentar os clientes com base em seus comportamentos de consumo e crédito.

Sugira uma estratégia de Data Understanding e Data Preparation.

Considere explicitamente as diretrizes de IA Responsável:
1. Minimização de Dados e Utilidade: Avalie o impacto de remover o identificador CUST_ID e como garantir que variáveis com ordens de grandeza muito diferentes (como limites de crédito de milhares de dólares vs. frequências de compra entre 0 e 1) colaborem de forma justa no cálculo das distâncias do K-Means.
2. Impacto de Decisões Automatizadas no Perfilamento: Discuta como a escolha de tratamento de outliers extremos (comuns em dados bancários) pode excluir ou invisibilizar perfis de clientes legítimos da estratégia final de negócios.

Ao sugerir a preparação, indique detalhadamente:
- possíveis atributos sensíveis (ou proxies de vulnerabilidade financeira, como a relação entre saldo devedor e pagamento mínimo);
- riscos de viés metodológico ou distorção dos centroides caso os dados não sejam padronizados corretamente;
- decisões de design dos dados que devem ser formalmente documentadas pelo grupo (ex: escolha entre StandardScaler ou RobustScaler);
- transformações necessárias (tratamento de valores nulos em MINIMUM_PAYMENTS e CREDIT_LIMIT, além de técnicas de redução de dimensionalidade como PCA, se aplicável);
- limitações estatísticas e os trade-offs do K-Means (como a necessidade de definir o número K previamente e a suposição de clusters esféricos).
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]


## 2.6 Comparação crítica — Data Understanding & Preparation

| Aspecto | Trilha A — baseline | Trilha B — guiada | Decisão do grupo |
|---|---|---|---|
| Tratamento de nulos | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Tratamento de outliers | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Atributos sensíveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Codificação de variáveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Redução de dimensionalidade | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Justificativa técnica | [ANALISAR] | [ANALISAR] | [DECISÃO] |
| Riscos e limitações | [ANALISAR] | [ANALISAR] | [DECISÃO] |

### Análise crítica

[INSERIR ANÁLISE COMPARATIVA]

A análise deve deixar claro:

- o que a LLM sugeriu corretamente;
- o que estava incompleto;
- o que estava tecnicamente errado;
- o que foi aceito;
- o que foi rejeitado;
- o que foi corrigido pelo grupo.


# 3. Modeling

Nesta seção, o grupo deve documentar a modelagem proposta com apoio da LLM.

Para o TP de padrões frequentes, a modelagem pode envolver, por exemplo:

- Apriori;
- FP-Growth;
- Eclat;
- regras de associação.

Para outros TPs, substituir pelos algoritmos pertinentes, como algoritmos de agrupamento ou classificação.

O foco da Fase 2 é comparar criticamente as soluções sugeridas pela LLM nas duas trilhas e, quando possível, executar testes preliminares para verificar a viabilidade das sugestões.


## 3.1 Interações com LLM — Modeling

### Subtarefa comparável

**Subtarefa escolhida:** [EXEMPLO: propor algoritmos, parâmetros, métricas e estratégia de avaliação para a tarefa]

---

### Trilha A — baseline

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — baseline

```text
Com base no dataset CC GENERAL e na tarefa de agrupamento com o algoritmo K-Means, proponha uma estratégia de modelagem.

Indique os algoritmos e abordagens necessárias para definir o número ideal de clusters (K), sugestões de parâmetros iniciais (como o método de inicialização dos centroides), métricas de avaliação interna para validar a qualidade dos agrupamentos e como o grupo deve comparar e interpretar os resultados obtnar.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — guiado

```text
Com base no dataset CC GENERAL e na tarefa de agrupamento com o algoritmo K-Means, proponha uma estratégia de modelagem.

Considere explicitamente as diretrizes de IA Responsável:
1. Interpretabilidade e Explicabilidade dos Agrupamentos: Como o K-Means agrupa os dados puramente por distância matemática, garanta que a definição e a caracterização final de cada cluster (ex: 'Clientes com alto endividamento') sejam explicáveis e fáceis de auditar, evitando rotulagens arbitrárias que gerem estigma aos clientes.
2. Robustez Metodológica e Reprodutibilidade: Discuta a importância da escolha da semente de inicialização (random_state) e do método de escolha dos centroides iniciais (como o k-means++) para assegurar que os agrupamentos gerados sejam estáveis e não mudem drasticamente a cada execução do modelo.

Na resposta, indique detalhadamente:
- algoritmos e métodos adequados para a modelagem e definição do número ideal de grupos (como o Método do Cotovelo e o Gráfico de Silhueta);
- parâmetros iniciais recomendados (n_init, max_iter, init);
- métricas de avaliação interna da qualidade dos clusters (Inércia e Coeficiente de Silhueta);
- riscos metodológicos comuns ao aplicar K-Means em dados bancários reais (como a convergência para mínimos locais devido à sensibilidade a outliers);
- impacto das diretrizes de IA responsável nas escolhas de modelagem (por exemplo, como limitar o número de variáveis ou aplicar PCA para manter os clusters interpretáveis);
- trade-offs entre o desempenho métrico do modelo, a interpretabilidade dos clusters pelo time de negócios, o custo computacional e a qualidade humana da solução gerada.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]


## 3.2 Comparação das soluções sugeridas pela LLM

| Elemento | Trilha A — baseline | Trilha B — guiada | Diferença concreta? | Comentário do grupo |
|---|---|---|---|---|
| Algoritmos sugeridos | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Parâmetros sugeridos | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Métricas sugeridas | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Preparação exigida | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Interpretabilidade | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Custo computacional | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Riscos ou limitações | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Aderência às diretrizes | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |

### Síntese

[INSERIR ANÁLISE COMPARATIVA]

A comparação deve destacar evidências, não impressões vagas.


## 3.4 Template de modelagem para agrupamento

Use esta subseção se o TP atual envolver agrupamento. Caso contrário, remova ou deixe claro que ela não foi utilizada.

> **Atenção:** em agrupamento, não há rótulo verdadeiro na maioria dos casos. A avaliação deve combinar métricas internas, análise visual quando possível e interpretação substantiva dos grupos.


In [ ]:
# Exemplo genérico para agrupamento com K-Means
# O grupo deve adaptar features, normalização e métricas conforme o dataset.

def preparar_matriz_numerica_para_modelagem(df: pd.DataFrame, colunas=None) -> pd.DataFrame:
    if colunas is None:
        X = df.select_dtypes(include=np.number).copy()
    else:
        X = df[colunas].copy()

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    return X

def executar_kmeans_basico(X: pd.DataFrame, n_clusters=3):
    try:
        from sklearn.preprocessing import StandardScaler
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score, davies_bouldin_score
    except ImportError:
        print("Biblioteca scikit-learn não instalada.")
        return None

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    modelo = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    labels = modelo.fit_predict(X_scaled)

    resultados = {
        "modelo": modelo,
        "labels": labels,
        "silhouette": silhouette_score(X_scaled, labels) if n_clusters > 1 else np.nan,
        "davies_bouldin": davies_bouldin_score(X_scaled, labels) if n_clusters > 1 else np.nan,
        "inertia": modelo.inertia_
    }

    return resultados

# Exemplo de uso:
# X = preparar_matriz_numerica_para_modelagem(df_prep)
# resultados_cluster = executar_kmeans_basico(X, n_clusters=3)
# resultados_cluster


## 3.6 Resultados preliminares da modelagem

[INSERIR RESULTADOS OBTIDOS APÓS EXECUÇÃO DO CÓDIGO]

Inclua tabelas e comentários sobre os resultados relevantes para a tarefa:

### Se a tarefa for padrões frequentes

- número de padrões encontrados;
- número de regras geradas;
- suporte, confiança e lift;
- tempo de execução;
- redundância e interpretabilidade das regras.

### Se a tarefa for agrupamento

- número de clusters;
- métricas internas, como Silhouette e Davies-Bouldin;
- tamanho dos grupos;
- interpretação dos clusters;
- estabilidade com diferentes parâmetros.

### Se a tarefa for classificação

- métricas como acurácia, precisão, revocação, F1-score e matriz de confusão;
- análise de classes minoritárias;
- overfitting ou underfitting;
- interpretabilidade e possíveis vieses.

### Atenção

Não conclua que um algoritmo é “melhor” apenas por uma métrica isolada.

Uma comparação adequada deve considerar:

- qualidade da solução;
- interpretabilidade;
- custo computacional;
- estabilidade;
- adequação ao problema;
- relação com as diretrizes escolhidas.


## 3.7 Análise da aderência às diretrizes

| Diretriz | Evidência na Trilha A | Evidência na Trilha B | A Trilha B melhorou? | Justificativa |
|---|---|---|---|---|
| [DIRETRIZ 1] | [PREENCHER] | [PREENCHER] | [SIM/NÃO/PARCIALMENTE] | [JUSTIFICAR] |
| [DIRETRIZ 2] | [PREENCHER] | [PREENCHER] | [SIM/NÃO/PARCIALMENTE] | [JUSTIFICAR] |

### Discussão

[INSERIR DISCUSSÃO]

A resposta deve ser específica. Exemplos de boa análise:

- “A Trilha B sugeriu registrar parâmetros e versões dos experimentos, o que melhora rastreabilidade.”
- “A Trilha B sugeriu avaliar desempenho por subgrupos, o que é pertinente à diretriz de justiça e viés.”
- “A Trilha B recomendou um algoritmo mais interpretável, mas com possível perda de desempenho.”
- “Apesar de mencionar privacidade, a Trilha B não propôs nenhuma alteração concreta no pipeline.”

Evite frases vagas como:

- “A resposta guiada foi mais responsável.”
- “A LLM considerou melhor as diretrizes.”
- “O resultado foi mais ético.”


# 4. Evaluation

Nesta seção, o grupo deve avaliar criticamente:

1. os resultados preliminares obtidos;
2. a utilidade das sugestões da LLM;
3. os erros e limitações das respostas;
4. as diferenças entre a Trilha A e a Trilha B;
5. os trade-offs observados;
6. a contribuição real das diretrizes de IA responsável.


## 4.1 Análise dos resultados

[INSERIR ANÁLISE DOS RESULTADOS]

A análise deve responder:

- Os padrões, clusters, classificações ou resultados encontrados fazem sentido?
- Eles são relevantes para o problema de negócio?
- Há resultados triviais, redundantes ou pouco úteis?
- O desempenho obtido é adequado?
- As métricas usadas são compatíveis com a tarefa?
- Há limitações nos dados ou na modelagem que afetam a conclusão?

> Se os experimentos não produziram bons resultados, isso também é um resultado válido, desde que seja analisado tecnicamente.


## 4.2 Avaliação das sugestões da LLM

| Sugestão da LLM | Trilha | Decisão do grupo | Justificativa |
|---|---|---|---|
| [SUGESTÃO 1] | Baseline | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 2] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 3] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |

### Erros, omissões ou alucinações identificadas

| Problema | Trilha | Por que é um problema? | Como o grupo corrigiu? |
|---|---|---|---|
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |

### Comentário crítico

[INSERIR COMENTÁRIO SOBRE A QUALIDADE DAS RESPOSTAS DA LLM]


## 4.3 Comparação final entre as trilhas

| Critério | Trilha A — baseline | Trilha B — guiada | Avaliação crítica |
|---|---|---|---|
| Utilidade prática | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Correção técnica | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Clareza das justificativas | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Aderência às diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da modelagem | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da avaliação | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Riscos não tratados | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Conclusão comparativa

[INSERIR CONCLUSÃO]

A conclusão deve indicar se a explicitação das diretrizes:

- produziu mudanças concretas;
- produziu apenas mudanças superficiais;
- não produziu diferença relevante;
- introduziu novos trade-offs;
- ajudou a identificar limitações ou riscos;
- levou o grupo a alterar decisões técnicas.


## 4.4 Trade-offs identificados

| Trade-off | Evidência | Decisão do grupo |
|---|---|---|
| Interpretabilidade × desempenho | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Privacidade × utilidade dos dados | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Custo computacional × qualidade da solução | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Simplicidade × completude da análise | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Justiça/viés × disponibilidade de atributos | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |

### Discussão

[INSERIR DISCUSSÃO SOBRE OS TRADE-OFFS]

Nem todo trade-off estará presente em todo trabalho. O grupo deve discutir apenas os que forem pertinentes ao dataset e à tarefa.


## 4.5 Considerações finais da Fase 2

[INSERIR CONSIDERAÇÕES FINAIS]

A conclusão da Fase 2 deve responder:

1. A LLM ajudou em quais partes do trabalho?
2. Em quais partes a LLM errou ou foi superficial?
3. A Trilha B foi de fato diferente da Trilha A?
4. As diretrizes escolhidas tiveram efeito concreto?
5. Quais sugestões serão levadas para a Fase 3?
6. Quais sugestões serão descartadas?
7. O que ainda precisa ser implementado ou corrigido na versão final?

### Próximos passos para a Fase 3

- [ ] Corrigir limitações identificadas nas respostas da LLM.
- [ ] Consolidar a preparação dos dados.
- [ ] Executar modelagem final.
- [ ] Avaliar resultados de forma mais completa.
- [ ] Integrar análise comparativa baseline × guiada × solução final.
- [ ] Revisar documentação e reprodutibilidade.


# 5. Checklist final da Fase 2

Antes de entregar, verifique se o notebook contém:

- [ ] identificação do grupo;
- [ ] link público do dataset;
- [ ] comprovação de que o dataset atende às restrições mínimas;
- [ ] descrição do problema de negócio;
- [ ] descrição das colunas;
- [ ] diretrizes de IA responsável escolhidas;
- [ ] justificativa das diretrizes;
- [ ] hipótese experimental;
- [ ] Trilha A — baseline;
- [ ] Trilha B — guiada;
- [ ] modelo utilizado em cada trilha;
- [ ] data de acesso em cada trilha;
- [ ] links das conversas;
- [ ] prompts principais;
- [ ] síntese crítica das respostas;
- [ ] comparação entre baseline e guiada;
- [ ] sugestões aceitas, rejeitadas e corrigidas;
- [ ] análise de erros e limitações da LLM;
- [ ] conexão com Business Understanding;
- [ ] conexão com Data Understanding & Preparation;
- [ ] conexão com Modeling;
- [ ] conexão com Evaluation;
- [ ] considerações finais e próximos passos.


# 6. Referências

[INSERIR REFERÊNCIAS USADAS PELO GRUPO]

Exemplos de referências possíveis:

- documentação do dataset;
- materiais da disciplina;
- documentação das bibliotecas usadas;
- referências sobre CRISP-DM;
- referências sobre IA responsável;
- referências sobre algoritmos aplicados.

> As referências devem apoiar decisões técnicas e metodológicas. Não use referências apenas de forma decorativa.


# Apêndice A — Modelo de registro de interações com LLM

Esta seção é opcional, mas recomendada para organizar evidências.

| ID | Trilha | Subtarefa | Modelo | Data | Link | Prompt resumido | Decisão do grupo |
|---|---|---|---|---|---|---|---|
| A1 | Baseline | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B1 | Guiada | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A2 | Baseline | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B2 | Guiada | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A3 | Baseline | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B3 | Guiada | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |


In [ ]:
# Apêndice B — Estrutura opcional para registrar prompts e respostas de forma organizada
# Esta estrutura pode ajudar o grupo a manter rastreabilidade.
# Evite inserir respostas muito longas no notebook; prefira sínteses críticas e links das conversas.

registros_llm = [
    {
        "id": "A1",
        "trilha": "baseline",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    },
    {
        "id": "B1",
        "trilha": "guiada",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT COM DIRETRIZES]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    }
]

df_registros_llm = pd.DataFrame(registros_llm)
display(df_registros_llm)
